[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rndsrc/orbits-py/blob/main/handson.ipynb)

# Speeding Up Your Python Codes 1000x

## Introduction

In this 45-minute workshop we will speed up a gravitational $n$-body simulation. How close can we get to a factor of 1000 on the machine running this notebook? We will measure each improvement against the same Python baseline.

We will work through four questions.

* How do we compute the force between bodies and advance their orbits with leapfrog?
* Where does Python spend its time in a single step?
* How much work can we save by reusing calculations, using NumPy arrays, or compiling the step with JAX?
* Does the optimized integrator bring three bodies back to their starting positions after one figure-eight orbit?

Work through the cells in order. At each TODO, replace `...` with your code, run the function cell, then run its check and timing. To use a worked solution, expand the reference cell, remove `_ref` from that function's name, and run the reference cell before continuing.

## The $n$-body Problem and the Leapfrog Algorithm

In a gravitational $n$-body simulation, each body accelerates under the attraction of all the others. The same pair interaction appears in calculations of [planetary orbits](https://rebound.readthedocs.io/en/latest/) and [cosmic structure formation](https://wwwmpa.mpa-garching.mpg.de/gadget/).

A direct calculation sums $n-1$ contributions for each of $n$ bodies, so doubling the number of bodies roughly quadruples the work. This makes the force calculation a useful place to study optimization. We will advance the orbits with leapfrog, which alternates velocity updates ("kicks") and position updates ("drifts").

Newton's law of gravitation gives us the force on each body. Newton's second law then gives us its acceleration.

### Gravitational Force and Acceleration

For two bodies of masses $m_i$ and $m_j$ at positions $\mathbf r_i$ and $\mathbf r_j$, the force on $i$ due to $j$ is
$$
  \mathbf f_{ij} = -G m_i m_j \frac{\mathbf r_i-\mathbf r_j}{|\mathbf r_i-\mathbf r_j|^3}.
$$
Here $G$ is the gravitational constant. The displacement in the numerator points from $j$ to $i$, so the minus sign gives an attractive force with inverse-square magnitude.

Summing the contributions from all other bodies gives the net force on body $i$,
\begin{align}
  \mathbf{f}_i
  = \sum_{j\ne i} \mathbf{f}_{ij}
  = -G m_i \sum_{j\ne i} m_j \frac{\mathbf{r}_i - \mathbf{r}_j}{|\mathbf{r}_i - \mathbf{r}_j|^3},
\end{align}

Dividing by $m_i$ gives the acceleration,
\begin{align}
  \mathbf{a}_i
  = \sum_{j\ne i} \mathbf{a}_{ij}
  = -G \sum_{j\ne i} m_j \frac{\mathbf{r}_i - \mathbf{r}_j}{|\mathbf{r}_i - \mathbf{r}_j|^3}.
\end{align}

We choose units with $G=1$ and use unsoftened point masses. Each kernel excludes self-interactions. A collision between two distinct bodies makes the force undefined, which we represent with `NaN` for those bodies.

In [ ]:
def acc1(m, r):
    
    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                axij = - m[j] * (xi - xj) / ((xi - xj)**2 + (yi - yj)**2 + (zi - zj)**2)**(3/2)
                ayij = - m[j] * (yi - yj) / ((xi - xj)**2 + (yi - yj)**2 + (zi - zj)**2)**(3/2)
                azij = - m[j] * (zi - zj) / ((xi - xj)**2 + (yi - yj)**2 + (zi - zj)**2)**(3/2)

                axi += axij
                ayi += ayij
                azi += azij

        a.append((axi, ayi, azi))

    return a

Two unit masses at $x=+1$ and $x=-1$ are two units apart. Their accelerations should point toward one another, with x-components $-1/4$ and $+1/4$, respectively.

In [ ]:
m  = [1.0, 1.0]
r0 = [
    (+1.0, 0.0, 0.0),
    (-1.0, 0.0, 0.0),
]
a0ref = [
    (-0.25, 0.0, 0.0),
    (+0.25, 0.0, 0.0),
]

In [ ]:
a0 = acc1(m, r0)

In [ ]:
a0

In [ ]:
import numpy as np

np.testing.assert_allclose(a0, a0ref)

### Leapfrog Integrator

For an isolated Newtonian system, the pair forces cancel in the total momentum balance. A leapfrog step preserves that cancellation up to roundoff. With a fixed, sufficiently small time step, leapfrog typically gives a bounded energy error over many orbits. Orbital phase can still accumulate error, so we will also check where each body returns after one period.

1. Half-step velocity update (Kick):
   start by updating the velocity by half a time step using the current acceleration $a(t)$:
   \begin{align}
     v\left(t+\frac{1}{2}\Delta t\right) = v(t) + \frac{1}{2}\Delta t\, a(t).
   \end{align}

2. Full-step position update (Drift):
   update the position for a full time step using the half-stepped velocity:
   \begin{align}
     x(t+\Delta t) = x(t) + \Delta t\, v\left(t+\frac{1}{2}\Delta t\right).
   \end{align}

3. Half-step velocity update (Kick):
   finally, update the velocity by another half time step using the new acceleration $a(t+\Delta t)$ computed from the updated positions:
   \begin{align}
     v(t+\Delta t) = v\left(t+\frac{1}{2}\Delta t\right) + \frac{1}{2}\Delta t\, a(t+\Delta t).
   \end{align}

The Python function below performs one complete kick–drift–kick step.

In [ ]:
def leapfrog1(m, r0, v0, dt, acc=acc1):

    n  = len(m)

    # vh = v0 + 0.5 * dt * a0
    a0 = acc(m, r0)
    vh = []
    for i in range(n):
        a0xi, a0yi, a0zi = a0[i]
        v0xi, v0yi, v0zi = v0[i]
        vhxi = v0xi + 0.5 * dt * a0xi
        vhyi = v0yi + 0.5 * dt * a0yi
        vhzi = v0zi + 0.5 * dt * a0zi
        vh.append((vhxi, vhyi, vhzi))

    # r1 = r0 + dt * vh
    r1 = []
    for i in range(n):
        vhxi, vhyi, vhzi = vh[i]
        x0i,  y0i,  z0i  = r0[i]
        x1i = x0i + dt * vhxi
        y1i = y0i + dt * vhyi
        z1i = z0i + dt * vhzi
        r1.append((x1i, y1i, z1i))

    # v1 = vh + 0.5 * dt * a1
    a1 = acc(m, r1)
    v1 = []
    for i in range(n):
        a1xi, a1yi, a1zi = a1[i]
        vhxi, vhyi, vhzi = vh[i]
        v1xi = vhxi + 0.5 * dt * a1xi
        v1yi = vhyi + 0.5 * dt * a1yi
        v1zi = vhzi + 0.5 * dt * a1zi
        v1.append((v1xi, v1yi, v1zi))

    return r1, v1

Give the two bodies opposite tangential velocities and take one step of length 0.1. We can check both the drifted positions and the final velocities.

In [ ]:
v0 = [
    (0.0, +0.5, 0.0),
    (0.0, -0.5, 0.0),
]
v1ref = [
    (-0.024984345739803245, +0.4993750014648409, 0.0),
    (+0.024984345739803245, -0.4993750014648409, 0.0),
]

In [ ]:
r1, v1 = leapfrog1(m, r0, v0, 0.1)

In [ ]:
v1

In [ ]:
np.testing.assert_allclose(r1, [[0.99875, 0.05, 0], [-0.99875, -0.05, 0]], rtol=1e-12)
np.testing.assert_allclose(v1, v1ref, rtol=1e-12)

### Test the Integrator

Each body has orbital radius 1 and tangential speed 0.5, so its circular period is $4\pi$. We integrate for one period in 128 steps. How close does each body come to its starting position?

In [ ]:
N = 128
T = 4 * np.pi

In [ ]:
dt = T / N
R  = [r0]
V  = [v0]
for _ in range(N):
    r, v = leapfrog1(m, R[-1], V[-1], dt)
    R.append(r)
    V.append(v)

NumPy lets us extract the x- and y-coordinates for every body at every time step.

In [ ]:
R = np.array(R)
X = R[:,:,0]
Y = R[:,:,1]

Plot both trajectories on the same spatial scale and measure the largest return-position error.

In [ ]:
from matplotlib import pyplot as plt

plt.plot(X, Y, '.-')
plt.gca().set_aspect('equal')
plt.xlabel("x")
plt.ylabel("y")
plt.show()
print("Largest return-position error:", np.linalg.norm(R[-1] - R[0], axis=1).max())

An animation lets us follow each body around the orbit. The progress bar tracks the frames as we build them.

In [ ]:
from matplotlib.animation import ArtistAnimation
from IPython.display import HTML
from tqdm.auto import tqdm

def animate(X, Y, ntail=10):
    X, Y = np.asarray(X), np.asarray(Y)
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.set(xlabel="x", ylabel="y", aspect="equal")

    frames = []
    for i in tqdm(range(len(X)), desc="Building animation", unit="frame"):
        start = max(0, i - ntail)
        ax.set_prop_cycle(None)
        lines = ax.plot(X[start:i + 1], Y[start:i + 1], ".-", animated=True)
        frames.append(lines)

    anim = ArtistAnimation(fig, frames, interval=50)
    plt.close(fig)
    return anim

In [ ]:
anim = animate(X, Y)
HTML(anim.to_jshtml())

## Timing a Leapfrog Step

We use the same 1,000 bodies and time step for every benchmark. IPython's `%timeit` measures one call five times. For complete steps, divide the mean time of `leapfrog1()` with `acc1()` by the mean time of the new step. For force evaluations, use one call to `acc1()` as the baseline.

In [ ]:
n = 1000
dt_bench = 0.001
rng = np.random.default_rng(42)
m_py = rng.lognormal(size=n).tolist()
r_py = rng.normal(size=(n, 3)).tolist()
v_py = rng.normal(size=(n, 3)).tolist()

In [ ]:
baseline = %timeit -n 1 -r 5 -o leapfrog1(m_py, r_py, v_py, dt_bench, acc=acc1)
print("Python step baseline (leapfrog1 with acc1), 1.0×")

force_baseline = %timeit -n 1 -r 5 -o acc1(m_py, r_py)
print("Python force baseline (acc1), 1.0×")

The original leapfrog step evaluates the force twice. The force-only timing measures one of those two evaluations.

### Where does the time go?

Profile the original step once to see how much time is spent evaluating forces.

In [ ]:
import cProfile

cProfile.run("leapfrog1(m_py, r_py, v_py, dt_bench, acc=acc1)", sort="tottime")

Compare the self time of `acc1` with that of `list.append`. A function can be called frequently and still account for little of the total runtime. Which part of the step would you change first?

## Optimization 1. Use List Comprehensions

A list comprehension can express each kick or drift as a single Python expression. The force calculation still does the same pairwise work, so the overall benefit depends on how much time the step spends updating the lists.

In [ ]:
iterable = range(10)

# Standard python for-loop
l1 = []
for x in iterable:
    l1.append(x * 2)

# Using a list comprehension
l2 = [x * 2 for x in iterable]

# Compare results
print(l1)
print(l2)

The first half-kick is written as a list comprehension. Follow its indexing to write the drift and the final half-kick. Compare the completed step with the Python step baseline, which uses the same force function.

In [ ]:
def leapfrog2(m, r0, v0, dt, acc=acc1):
    n = len(m)
    a0 = acc(m, r0)
    vh = [[v0[i][k] + 0.5 * dt * a0[i][k] for k in range(3)]
          for i in range(n)]

    # TODO: Drift each position for a full step using vh. Use a list comprehension.
    r1 = ...

    a1 = acc(m, r1)
    # TODO: Apply the final half-kick to vh using a1, again with a list comprehension.
    v1 = ...
    return r1, v1

In [ ]:
# @title Reference for leapfrog2 { display-mode: "form" }
def leapfrog2_ref(m, r0, v0, dt, acc=acc1):
    n = len(m)
    a0 = acc(m, r0)
    vh = [[v0[i][k] + 0.5 * dt * a0[i][k] for k in range(3)]
          for i in range(n)]

    r1 = [[r0[i][k] + dt * vh[i][k] for k in range(3)]
          for i in range(n)]

    a1 = acc(m, r1)
    v1 = [[vh[i][k] + 0.5 * dt * a1[i][k] for k in range(3)]
          for i in range(n)]
    return r1, v1

In [ ]:
# Compare one binary step with the result from leapfrog1.
np.testing.assert_allclose(leapfrog2(m, r0, v0, 0.1), (r1, v1))
step_timing = %timeit -n 1 -r 5 -o leapfrog2(m_py, r_py, v_py, dt_bench, acc=acc1)
print(f"Speedup over Python step baseline (leapfrog1 with acc1): {baseline.average / step_timing.average:.1f}×")

## Optimization 2. Reduce the Operation Count

The three components of an acceleration share the same separation and mass factor. Reusing these quantities inside the pair loop saves work every time we evaluate the force.

As we change the force kernel, we can check each version on three unequal masses at $x=-1,0,1$. This gives each body two contributions to sum. We will use the direct Python calculation as our reference.

In [ ]:
m_test = [1.0, 2.0, 4.0]
r_test = [[-1.0, 0.0, 0.0], [0.0, 0.0, 0.0], [1.0, 0.0, 0.0]]
a_test = acc1(m_test, r_test)

In `acc1()`, each coordinate uses the same denominator. In `acc2()`, compute $|\mathbf r_i-\mathbf r_j|^3$ once per pair and use it in all three components.

In [ ]:
def acc2(m, r):
    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                # TODO: Compute the cubed distance once for this pair.
                rrr = ...

                # TODO: Use the same rrr for each acceleration component.
                axij = ...
                ayij = ...
                azij = ...

                axi += axij
                ayi += ayij
                azi += azij

        a.append((axi, ayi, azi))

    return a

In [ ]:
# @title Reference for acc2 { display-mode: "form" }
def acc2_ref(m, r):

    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                # "Cache" r^3
                rrr = ((xi - xj)**2 + (yi - yj)**2 + (zi - zj)**2)**(3/2)

                axij = - m[j] * (xi - xj) / rrr
                ayij = - m[j] * (yi - yj) / rrr
                azij = - m[j] * (zi - zj) / rrr

                axi += axij
                ayi += ayij
                azi += azij

        a.append((axi, ayi, azi))

    return a

In [ ]:
np.testing.assert_allclose(acc2(m_test, r_test), a_test)
force_time = %timeit -n 1 -r 5 -o acc2(m_py, r_py)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

How much time did reusing the denominator save per force evaluation?

In `acc3()`, look for coordinate differences that appear both in the cubed distance and in the numerator. Store each difference once and use it in both places.

In [ ]:
def acc3(m, r):
    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                # TODO: Store the three components of r[i] - r[j].
                dx = ...
                dy = ...
                dz = ...

                # TODO: Use the stored components to compute the cubed distance.
                rrr = ...

                # TODO: Use the stored differences in the numerators as well.
                axij = ...
                ayij = ...
                azij = ...

                axi += axij
                ayi += ayij
                azi += azij

        a.append((axi, ayi, azi))

    return a

In [ ]:
# @title Reference for acc3 { display-mode: "form" }
def acc3_ref(m, r):

    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                # "Cache" the components of dr = r_ij = ri - rj
                dx = xi - xj
                dy = yi - yj
                dz = zi - zj

                rrr = (dx**2 + dy**2 + dz**2)**(3/2)

                axij = - m[j] * dx / rrr
                ayij = - m[j] * dy / rrr
                azij = - m[j] * dz / rrr

                axi += axij
                ayi += ayij
                azi += azij

        a.append((axi, ayi, azi))

    return a

In [ ]:
# Use positions that differ in all three coordinates.
r_test_3d = [[0., 0., 0.], [1., 2., 3.], [-2., 1., -1.]]
np.testing.assert_allclose(acc3(m_test, r_test_3d), acc1(m_test, r_test_3d))
force_time = %timeit -n 1 -r 5 -o acc3(m_py, r_py)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

The three components in `acc3()` also share a mass and a denominator. In `acc4()`, combine them into one signed factor per pair.

In [ ]:
def acc4(m, r):
    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                dx = xi - xj
                dy = yi - yj
                dz = zi - zj

                rrr = (dx**2 + dy**2 + dz**2)**(3/2)
                # TODO: Compute -m[j] / rrr once, then scale dx, dy and dz.
                f = ...
                axi += ...
                ayi += ...
                azi += ...

        a.append((axi, ayi, azi))

    return a

In [ ]:
# @title Reference for acc4 { display-mode: "form" }
def acc4_ref(m, r):

    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                dx = xi - xj
                dy = yi - yj
                dz = zi - zj

                rrr = (dx**2 + dy**2 + dz**2)**(3/2)
                f   = - m[j] / rrr # "cache" -m_j / r_{ij}^3

                axi += f * dx
                ayi += f * dy
                azi += f * dz

        a.append((axi, ayi, azi))

    return a

In [ ]:
np.testing.assert_allclose(acc4(m_test, r_test_3d), acc1(m_test, r_test_3d))
force_time = %timeit -n 1 -r 5 -o acc4(m_py, r_py)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

Newton's third law gives $\mathbf f_{ij}=-\mathbf f_{ji}$, or $m_i\mathbf a_{ij}=-m_j\mathbf a_{ji}$. With unequal masses the two accelerations have different magnitudes. To update both bodies from one pair, we first give each body its own mutable row in the acceleration list. Keep the pair loop from `acc4` for this exercise so you can measure the effect of changing how the accelerations are stored.

In [ ]:
def acc5(m, r):
    n = len(m)
    # TODO: Give each body its own row of three zeros.
    a = ...
    for i in range(n):
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                dx = xi - xj
                dy = yi - yj
                dz = zi - zj

                rrr = (dx**2 + dy**2 + dz**2)**(3/2)
                f   = - m[j] / rrr

                # TODO: Add this pair's contribution directly to the row for body i.
                a[i][0] += ...
                a[i][1] += ...
                a[i][2] += ...

    return a

In [ ]:
# @title Reference for acc5 { display-mode: "form" }
def acc5_ref(m, r):

    n = len(m)
    a = [[0.0, 0.0, 0.0] for _ in range(n)]
    for i in range(n):
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                dx = xi - xj
                dy = yi - yj
                dz = zi - zj

                rrr = (dx**2 + dy**2 + dz**2)**(3/2)
                f   = - m[j] / rrr

                # Accumulate in the row for body i.
                a[i][0] += f * dx
                a[i][1] += f * dy
                a[i][2] += f * dz

    return a

In [ ]:
np.testing.assert_allclose(acc5(m_test, r_test_3d), acc1(m_test, r_test_3d))
force_time = %timeit -n 1 -r 5 -o acc5(m_py, r_py)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

Now visit each unordered pair once by looping over $j<i$. Add the two mass-weighted contributions to their respective rows. This halves the number of pair evaluations, although each evaluation now updates two bodies.

In [ ]:
def acc6(m, r):
    n = len(m)
    a = [[0.0, 0.0, 0.0] for _ in range(n)]
    for i in range(n):
        # TODO: Choose the range so each pair is visited once, with j < i.
        for j in ...:
            xi, yi, zi = r[i]
            xj, yj, zj = r[j]

            dx = xi - xj
            dy = yi - yj
            dz = zi - zj

            rrr = (dx**2 + dy**2 + dz**2)**(3/2)
            # TODO: Choose the mass and sign for each body's contribution.
            fi = ...
            fj = ...

            # TODO: Accumulate the three components for each body.
            a[i][0] += ...
            a[i][1] += ...
            a[i][2] += ...

            a[j][0] += ...
            a[j][1] += ...
            a[j][2] += ...

    return a

In [ ]:
# @title Reference for acc6 { display-mode: "form" }
def acc6_ref(m, r):

    n = len(m)
    a = [[0.0, 0.0, 0.0] for _ in range(n)]
    for i in range(n):
        for j in range(i): # adjust the upper bound of the inner loop
            xi, yi, zi = r[i]
            xj, yj, zj = r[j]

            dx = xi - xj
            dy = yi - yj
            dz = zi - zj

            rrr = (dx**2 + dy**2 + dz**2)**(3/2)
            fi  =   m[i] / rrr
            fj  = - m[j] / rrr

            a[i][0] += fj * dx
            a[i][1] += fj * dy
            a[i][2] += fj * dz

            # Update the acceleration of body j.
            a[j][0] += fi * dx
            a[j][1] += fi * dy
            a[j][2] += fi * dz

    return a

In [ ]:
np.testing.assert_allclose(acc6(m_test, r_test_3d), acc1(m_test, r_test_3d))
force_time = %timeit -n 1 -r 5 -o acc6(m_py, r_py)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

The acceleration at the end of one step is also the acceleration at the start of the next. In `leapfrog3()`, use the supplied `a0` for the first kick and return the new acceleration along with the position and velocity.

In [ ]:
def leapfrog3(m, r0, v0, a0, dt, acc=acc6):
    n = len(m)
    # TODO: Use the supplied a0 for the first half-kick.
    vh = ...

    r1 = [[r0[i][k] + dt * vh[i][k] for k in range(3)]
          for i in range(n)]

    a1 = acc(m, r1)
    v1 = [[vh[i][k] + 0.5 * dt * a1[i][k] for k in range(3)]
          for i in range(n)]
    # TODO: Return the new position, velocity and acceleration for the next step.
    return ...

In [ ]:
# @title Reference for leapfrog3 { display-mode: "form" }
def leapfrog3_ref(m, r0, v0, a0, dt, acc=acc6):
    n = len(m)
    vh = [[v0[i][k] + 0.5 * dt * a0[i][k] for k in range(3)]
          for i in range(n)]

    r1 = [[r0[i][k] + dt * vh[i][k] for k in range(3)]
          for i in range(n)]

    a1 = acc(m, r1)
    v1 = [[vh[i][k] + 0.5 * dt * a1[i][k] for k in range(3)]
          for i in range(n)]
    return r1, v1, a1

In [ ]:
# A cached step also returns the acceleration to use on the next step.
cached_step = leapfrog3(m, r0, v0, a0, 0.1, acc=acc4)
np.testing.assert_allclose(cached_step, (r1, v1, acc1(m, r1)))

a_py = acc4(m_py, r_py)
step_timing = %timeit -n 1 -r 5 -o leapfrog3(m_py, r_py, v_py, a_py, dt_bench, acc=acc4)
print(f"Speedup over Python step baseline (leapfrog1 with acc1): {baseline.average / step_timing.average:.1f}×")

In [ ]:
a_py = acc6(m_py, r_py)
step_timing = %timeit -n 1 -r 5 -o leapfrog3(m_py, r_py, v_py, a_py, dt_bench, acc=acc6)
print(f"Speedup over Python step baseline (leapfrog1 with acc1): {baseline.average / step_timing.average:.1f}×")

Both step timings use the initial acceleration computed before the timer starts. The first uses `acc4()` and the second uses `acc6()`. Over an orbit of $N$ steps, carrying the acceleration between steps reduces the number of force evaluations from $2N$ to $N+1$, including the initial evaluation.

## Optimization 3. Compare Multiplication and Exponentiation

For a floating-point displacement, we can calculate its square with either `dx * dx` or `dx**2`. Their relative cost depends on the Python implementation and on the surrounding calculation.

Start with the all-pairs loop from `acc4()`. In `acc7()`, change only how the three displacement components are squared.

In [ ]:
def acc7(m, r):
    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                dx = xi - xj
                dy = yi - yj
                dz = zi - zj

                # TODO: Replace dx**2, dy**2, and dz**2 with explicit multiplication.
                rrr = ...
                f = -m[j] / rrr

                axi += f * dx
                ayi += f * dy
                azi += f * dz

        a.append((axi, ayi, azi))

    return a

In [ ]:
# @title Reference for acc7 { display-mode: "form" }
def acc7_ref(m, r): # this is a modification of acc4()

    n = len(m)
    a = []
    for i in range(n):
        axi, ayi, azi = 0, 0, 0
        for j in range(n):
            if j != i:
                xi, yi, zi = r[i]
                xj, yj, zj = r[j]

                dx = xi - xj
                dy = yi - yj
                dz = zi - zj

                rrr = (dx*dx + dy*dy + dz*dz)**(3/2) # replace dx**2 by dx*dx etc
                f   = - m[j] / rrr

                axi += f * dx
                ayi += f * dy
                azi += f * dz

        a.append((axi, ayi, azi))

    return a

In [ ]:
np.testing.assert_allclose(acc7(m_test, r_test_3d), acc1(m_test, r_test_3d))
force_time = %timeit -n 1 -r 5 -o acc7(m_py, r_py)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

Apply the same change to `acc6()`, which visits each pair once. Keep the updates to both bodies when you write `acc8()`.

In [ ]:
def acc8(m, r):
    n = len(m)
    a = [[0.0, 0.0, 0.0] for _ in range(n)]
    for i in range(n):
        for j in range(i):
            xi, yi, zi = r[i]
            xj, yj, zj = r[j]

            dx = xi - xj
            dy = yi - yj
            dz = zi - zj

            # TODO: Replace dx**2, dy**2, and dz**2 with explicit multiplication.
            rrr = ...
            fi = m[i] / rrr
            fj = -m[j] / rrr

            a[i][0] += fj * dx
            a[i][1] += fj * dy
            a[i][2] += fj * dz

            # Update the acceleration of body j.
            a[j][0] += fi * dx
            a[j][1] += fi * dy
            a[j][2] += fi * dz

    return a

In [ ]:
# @title Reference for acc8 { display-mode: "form" }
def acc8_ref(m, r): # this is a modification of acc6()

    n = len(m)
    a = [[0.0, 0.0, 0.0] for _ in range(n)]
    for i in range(n):
        for j in range(i):
            xi, yi, zi = r[i]
            xj, yj, zj = r[j]

            dx = xi - xj
            dy = yi - yj
            dz = zi - zj

            rrr = (dx*dx + dy*dy + dz*dz)**(3/2) # replace dx**2 by dx*dx etc
            fi  =   m[i] / rrr
            fj  = - m[j] / rrr

            a[i][0] += fj * dx
            a[i][1] += fj * dy
            a[i][2] += fj * dz

            a[j][0] += fi * dx
            a[j][1] += fi * dy
            a[j][2] += fi * dz

    return a

In [ ]:
np.testing.assert_allclose(acc8(m_test, r_test_3d), acc1(m_test, r_test_3d))
force_time = %timeit -n 1 -r 5 -o acc8(m_py, r_py)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

Which change made the larger difference on this machine, visiting each pair once or using explicit multiplication?

Next we will keep the same kick–drift–kick algorithm and express the force calculation in terms of arrays.

## Optimization 4. Use NumPy Arrays

Broadcasting gives us all pair displacements in an array of shape $(n,n,3)$. The last axis holds vector components and the second axis enumerates source bodies. The pair arrays use memory proportional to $n^2$, which limits the size of this direct implementation.

In [ ]:
m_np = np.asarray(m_py, dtype=np.float64)
r_np = np.asarray(r_py, dtype=np.float64)
v_np = np.asarray(v_py, dtype=np.float64)

In [ ]:
def acc9(m, r):
    # TODO: Broadcast r to form all r[i] - r[j] displacements with shape (n, n, 3).
    dr = ...
    # TODO: Sum the squared components along the coordinate axis.
    rr = ...
    # Exclude self before division. A distinct body at zero distance is singular.
    rr = np.where(np.eye(len(m), dtype=bool), np.inf, rr)
    rr = np.where(rr == 0, np.nan, rr)

    # TODO: Form the unit displacement and scale it by the signed source mass over rr.
    direction = ...
    magnitude = ...
    # TODO: Sum the pair accelerations over source bodies j to get an (n, 3) array.
    return ...

In [ ]:
# @title Reference for acc9 { display-mode: "form" }
def acc9_ref(m, r):
    dr = r[:, None, :] - r[None, :, :]
    rr = np.sum(dr * dr, axis=-1)
    # Exclude self before division. A distinct body at zero distance is singular.
    rr = np.where(np.eye(len(m), dtype=bool), np.inf, rr)
    rr = np.where(rr == 0, np.nan, rr)
    direction = dr / np.sqrt(rr)[:, :, None]
    magnitude = -m[None, :] / rr
    return np.sum(magnitude[:, :, None] * direction, axis=1)

In [ ]:
np.testing.assert_allclose(acc9(np.array(m_test), np.array(r_test_3d)), acc1(m_test, r_test_3d))
force_time = %timeit -n 1 -r 5 -o acc9(m_np, r_np)
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

Now express the kicks and the drift as operations on whole arrays. Keep the force function as an argument so the same stepper can also use JAX arrays.

In [ ]:
def leapfrog4(m, r0, v0, a0, dt, acc=acc9):
    ht = 0.5 * dt

    # TODO: Apply the first half-kick and the full drift to the whole arrays.
    vh = ...
    r1 = ...

    a1 = acc(m, r1)
    # TODO: Apply the final half-kick to vh using a1.
    v1 = ...
    return r1, v1, a1

In [ ]:
# @title Reference for leapfrog4 { display-mode: "form" }
def leapfrog4_ref(m, r0, v0, a0, dt, acc=acc9):
    ht = 0.5 * dt

    vh = v0 + ht * a0
    r1 = r0 + dt * vh

    a1 = acc(m, r1)
    v1 = vh + ht * a1

    return r1, v1, a1

In [ ]:
array_step = leapfrog4(np.array(m), np.array(r0), np.array(v0), np.array(a0), 0.1, acc=acc9)
np.testing.assert_allclose(array_step, (r1, v1, acc1(m, r1)))

a_np = acc9(m_np, r_np)
numpy64 = %timeit -n 1 -r 5 -o leapfrog4(m_np, r_np, v_np, a_np, dt_bench, acc=acc9)
print(f"Speedup over Python step baseline (leapfrog1 with acc1): {baseline.average / numpy64.average:.1f}×")

The timing above includes the NumPy updates and one force evaluation. Compare it with a Python step that also carries the initial acceleration.

## Optimization 5. Use Lower Precision

A float32 array uses half the storage of a float64 array with the same shape. Convert the inputs to single precision, check the three-body force, and compare the step time with the float64 NumPy step above. Both speedups use the original Python step as their baseline.

In [ ]:
def to_single(values):
    # TODO: Set the dtype to np.float32 for single precision.
    return np.asarray(values, dtype=...)

In [ ]:
# @title Reference for to_single { display-mode: "form" }
def to_single_ref(values):
    return np.asarray(values, dtype=np.float32)

In [ ]:
m_input, r_input, v_input = (to_single(x) for x in (m_np, r_np, v_np))
assert m_input.dtype == r_input.dtype == v_input.dtype == np.float32, "All three inputs should be float32."
np.testing.assert_allclose(acc9(to_single(m_test), to_single(r_test)), a_test)

a_input = acc9(m_input, r_input)
step_time = %timeit -n 1 -r 5 -o leapfrog4(m_input, r_input, v_input, a_input, dt_bench, acc=acc9)
print(f"Speedup over Python step baseline (leapfrog1 with acc1): {baseline.average / step_time.average:.1f}×")

## Optimization 6. Run the Array Calculation with JAX

JAX can run array operations on a CPU or a supported accelerator. Transfer the arrays from `to_single()` to the device before timing. Compare the NumPy and JAX input dtypes when you compare their step times. JAX can return control to Python while the device is still working, so each timed call must wait for all its output arrays. [JAX benchmarking guide](https://docs.jax.dev/en/latest/benchmarking.html)

In [ ]:
import jax
from jax import numpy as jnp

print("JAX", jax.__version__, "on", jax.devices()[0])

In [ ]:
m_jax = jnp.asarray(m_input)
r_jax = jnp.asarray(r_input)
v_jax = jnp.asarray(v_input)
print(f"JAX input dtypes   m={m_jax.dtype}, r={r_jax.dtype}, v={v_jax.dtype}")

In [ ]:
def acc10(m, r):
    dr = r[:, None, :] - r[None, :, :]
    # TODO: Port the squared-distance reduction in acc9 to jax.numpy (jnp).
    rr = ...
    # TODO: Use jnp to set the self-pair distances on the diagonal to infinity.
    rr = ...
    # TODO: Mark remaining zero distances between distinct bodies with NaN.
    rr = ...
    # TODO: Form the unit displacement using the JAX square root.
    direction = ...
    magnitude = -m[None, :] / rr
    # TODO: Use a JAX reduction to sum the pair accelerations over source bodies.
    return ...

In [ ]:
# @title Reference for acc10 { display-mode: "form" }
def acc10_ref(m, r):
    dr = r[:, None, :] - r[None, :, :]
    rr = jnp.sum(dr * dr, axis=-1)
    # Exclude self before division. A distinct body at zero distance is singular.
    rr = jnp.where(jnp.eye(len(m), dtype=bool), jnp.inf, rr)
    rr = jnp.where(rr == 0, jnp.nan, rr)
    direction = dr / jnp.sqrt(rr)[:, :, None]
    magnitude = -m[None, :] / rr
    return jnp.sum(magnitude[:, :, None] * direction, axis=1)

In [ ]:
np.testing.assert_allclose(acc10(jnp.asarray(m_test), jnp.asarray(r_test_3d)),
                           acc1(m_test, r_test_3d), rtol=1e-6)

a_jax = jax.block_until_ready(acc10(m_jax, r_jax))
force_time = %timeit -n 1 -r 5 -o jax.block_until_ready(acc10(m_jax, r_jax))
print(f"Speedup over Python force baseline (acc1): {force_baseline.average / force_time.average:.1f}×")

In [ ]:
jax_args = (m_jax, r_jax, v_jax, a_jax, dt_bench)
eager_step = jax.block_until_ready(leapfrog4(*jax_args, acc=acc10))

jax_eager = %timeit -n 1 -r 5 -o jax.block_until_ready(leapfrog4(*jax_args, acc=acc10))
print(f"Speedup over Python step baseline (leapfrog1 with acc1): {baseline.average / jax_eager.average:.1f}×")

The first timing measures one force evaluation. The second measures a full leapfrog step with the same force function. Both timings wait for the result and include any transfers between NumPy and JAX.

## Optimization 7. Compile the Step with JAX

The preceding timing ran `leapfrog4()` with your JAX force function. Compile the same array updates in `leapfrog5()` and keep the force function fixed for each compilation. The first call can include compilation, so we time it separately from the repeated calls.

In [ ]:
from time import perf_counter

In [ ]:
def leapfrog5(m, r0, v0, a0, dt, acc=acc10):
    ht = 0.5 * dt

    vh = v0 + ht * a0
    r1 = r0 + dt * vh

    a1 = acc(m, r1)
    v1 = vh + ht * a1

    return r1, v1, a1

# TODO: Compile this function with jax.jit, naming "acc" in static_argnames.
leapfrog5 = ...

In [ ]:
# @title Reference for leapfrog5 { display-mode: "form" }
def leapfrog5_ref(m, r0, v0, a0, dt, acc=acc10):
    ht = 0.5 * dt

    vh = v0 + ht * a0
    r1 = r0 + dt * vh

    a1 = acc(m, r1)
    v1 = vh + ht * a1

    return r1, v1, a1

leapfrog5_ref = jax.jit(leapfrog5_ref, static_argnames=("acc",))

In [ ]:
start = perf_counter()
first_step = jax.block_until_ready(leapfrog5(*jax_args, acc=acc10))
first_seconds = perf_counter() - start
np.testing.assert_allclose(first_step, eager_step, rtol=1e-4, atol=1e-4)
print(f"First completed call at n={n} took {first_seconds:.4f} s "
      f"({baseline.average / first_seconds:.1f}× vs. the Python step baseline)")

step5_time = %timeit -n 1 -r 5 -o jax.block_until_ready(leapfrog5(*jax_args, acc=acc10))
print(f"Speedup over Python step baseline (leapfrog1 with acc1): {baseline.average / step5_time.average:.1f}×")

After compiling `leapfrog5()`, compare the measured speedup with the original goal. The Python baseline evaluates the force twice, while `leapfrog5()` carries the acceleration between steps. With float32 inputs, the speedup over that baseline also includes the change in precision.

## Applying the $n$-Body Integrator

Each row of the initial-condition file gives one body's mass, position and velocity. Compute the initial acceleration with `acc10()`, then pass it to `leapfrog5()`. Each step returns the position, velocity and acceleration for the next step.

In [ ]:
def integrate(fname, T, N):
    if not isinstance(N, (int, np.integer)) or N < 1:
        raise ValueError("N must be a positive integer")
    ic = np.loadtxt(fname, ndmin=2)
    if ic.shape[1] != 7 or not np.isfinite(ic).all():
        raise ValueError("Expected seven finite values per body: m x y z vx vy vz")
    ic = jnp.asarray(ic, dtype=jnp.float32)
    m = ic[:, 0]
    r = ic[:, 1:4]
    v = ic[:, 4:7]
    a = acc10(m, r)
    R, V, A = [r], [v], [a]

    dt = T / N
    for _ in tqdm(range(N), desc="Integrating orbit", unit="step"):
        r, v, a = leapfrog5(m, r, v, a, dt, acc=acc10)
        R.append(r)
        V.append(v)
        A.append(a)

    R, V, A = jnp.stack(R), jnp.stack(V), jnp.stack(A)
    if not all(bool(jnp.isfinite(x).all()) for x in (R, V, A)):
        raise ValueError("Nonfinite orbit. Check for a collision or an unstable time step.")
    return jnp.linspace(0, T, N + 1), R, V, A

Three equal masses can follow one another around the same figure eight. For the supplied initial conditions, the published period is 6.32591398. We will measure how far each body ends up from its own starting position after one period. [Chenciner and Montgomery, Figure 1](https://perso.imcce.fr/alain-chenciner/huit.pdf)

Use the bundled `ic/figure-8.tsv`. If you opened the notebook on its own, this cell downloads the same file into `ic/`.

In [ ]:
from pathlib import Path
from subprocess import check_output

figure8_file = Path("ic/figure-8.tsv")
if not figure8_file.is_file():
    url = "https://raw.githubusercontent.com/rndsrc/orbits-py/167c1b79ee6416752ed5cfd28ce55de6051eb58e/ic/figure-8.tsv"
    figure8_file.parent.mkdir(exist_ok=True)
    figure8_file.write_bytes(check_output(["curl", "-fsSL", url], timeout=30))

In [ ]:
period = 6.32591398
n_steps = 512
times8, R8, V8, A8 = jax.block_until_ready(integrate(figure8_file, period, n_steps))
return_errors = np.linalg.norm(R8[-1] - R8[0], axis=1)
print("Largest return-position error:", return_errors.max())

In [ ]:
X8 = R8[:, :, 0]
Y8 = R8[:, :, 1]
plt.plot(X8, Y8, "-")
plt.gca().set(xlabel="x", ylabel="y", aspect="equal")
plt.show()

In [ ]:
# Show every eighth integration step to keep the figure-eight animation short.
anim8 = animate(X8[::8], Y8[::8])
HTML(anim8.to_jshtml())

## Conclusion and Discussion

Which optimization saved the most time on your machine? Compare timings at the same precision and with the same number of force evaluations to isolate one change. The overall speedup tells us what the changes accomplished together.

For the figure eight, the return-position error measures something the timing cannot. Double the number of steps and rerun the orbit. How much does the return-position error fall when you halve the time step?